In [1]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from torch.optim import AdamW

In [3]:
from google.colab import drive, runtime
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/rocky-chatbot/
!ls

/content/drive/MyDrive/rocky-chatbot
 augmented_train.json   rocky-flan-t5	   rocky-t5
 dialogue-t5	        rocky-only-t5	   val.json
'dialogue train'        rocky_only_train   with_context


In [11]:
MODEL_NAME     = "t5-base"
TRAIN_PATH     = "augmented_train.json"
VAL_PATH       = "val.json"
OUTPUT_DIR     = "dialogue-augmented-t5"
MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 64
BATCH_SIZE     = 8
EPOCHS         = 5
LEARNING_RATE  = 5e-5
WARMUP_STEPS   = 50
SAVE_EVERY     = 1      # save checkpoint every N epochs

In [12]:
class RockyDataset(Dataset):
    def __init__(self, path, tokenizer, max_input_len, max_target_len):
        with open(path, encoding="utf-8") as f:
            self.pairs = json.load(f)
        self.tokenizer     = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]

        inputs = self.tokenizer(
            pair["input_text"],
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        targets = self.tokenizer(
            pair["target_text"],
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Replace padding token id with -100 so it's ignored in loss
        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels":         labels,
        }

In [13]:
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    print(f"Loading {MODEL_NAME}...")
    tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
    model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.to(device)

    print("Loading datasets...")
    train_dataset = RockyDataset(TRAIN_PATH, tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
    val_dataset   = RockyDataset(VAL_PATH,   tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
    print(f"  {len(train_dataset)} train / {len(val_dataset)} val")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_steps,
    )

    best_val_loss = float("inf")

    for epoch in range(1, EPOCHS + 1):
        # --- Train ---
        model.train()
        train_loss = 0.0
        for step, batch in enumerate(train_loader):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            train_loss += loss.item()

            if (step + 1) % 10 == 0:
                avg = train_loss / (step + 1)
                print(f"  Epoch {epoch} step {step+1}/{len(train_loader)} loss={avg:.4f}")

        avg_train_loss = train_loss / len(train_loader)

        # --- Validate ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels         = batch["labels"].to(device)
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                val_loss += outputs.loss.item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"\nEpoch {epoch} — train loss: {avg_train_loss:.4f} | val loss: {avg_val_loss:.4f}\n")

        # --- Save ---
        if epoch % SAVE_EVERY == 0:
            checkpoint_dir = f"{OUTPUT_DIR}/checkpoint-epoch-{epoch}"
            model.save_pretrained(checkpoint_dir)
            tokenizer.save_pretrained(checkpoint_dir)
            print(f"Saved checkpoint to {checkpoint_dir}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            model.save_pretrained(f"{OUTPUT_DIR}/best")
            tokenizer.save_pretrained(f"{OUTPUT_DIR}/best")
            print(f"New best val loss {best_val_loss:.4f} — saved to {OUTPUT_DIR}/best")

    print("\nTraining complete.")
    print(f"Best val loss: {best_val_loss:.4f}")
    print(f"Best model saved to {OUTPUT_DIR}/best")

In [14]:
train()

Using device: cuda
Loading t5-base...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading datasets...
  6817 train / 75 val
  Epoch 1 step 10/853 loss=6.5336
  Epoch 1 step 20/853 loss=6.2756
  Epoch 1 step 30/853 loss=6.0398
  Epoch 1 step 40/853 loss=5.8373
  Epoch 1 step 50/853 loss=5.6774
  Epoch 1 step 60/853 loss=5.5112
  Epoch 1 step 70/853 loss=5.4014
  Epoch 1 step 80/853 loss=5.2971
  Epoch 1 step 90/853 loss=5.1871
  Epoch 1 step 100/853 loss=5.0955
  Epoch 1 step 110/853 loss=5.0061
  Epoch 1 step 120/853 loss=4.9443
  Epoch 1 step 130/853 loss=4.8714
  Epoch 1 step 140/853 loss=4.8043
  Epoch 1 step 150/853 loss=4.7597
  Epoch 1 step 160/853 loss=4.7074
  Epoch 1 step 170/853 loss=4.6699
  Epoch 1 step 180/853 loss=4.6303
  Epoch 1 step 190/853 loss=4.5926
  Epoch 1 step 200/853 loss=4.5492
  Epoch 1 step 210/853 loss=4.5111
  Epoch 1 step 220/853 loss=4.4881
  Epoch 1 step 230/853 loss=4.4672
  Epoch 1 step 240/853 loss=4.4412
  Epoch 1 step 250/853 loss=4.4031
  Epoch 1 step 260/853 loss=4.3720
  Epoch 1 step 270/853 loss=4.3535
  Epoch 1 step 280/853

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint to dialogue-augmented-t5/checkpoint-epoch-1


...

Saved checkpoint to dialogue-augmented-t5/checkpoint-epoch-3
  Epoch 4 step 10/853 loss=1.7671
  Epoch 4 step 20/853 loss=1.7622
  Epoch 4 step 30/853 loss=1.8078
  Epoch 4 step 40/853 loss=1.8328
  Epoch 4 step 50/853 loss=1.8170
  Epoch 4 step 60/853 loss=1.8279
  Epoch 4 step 70/853 loss=1.8128
  Epoch 4 step 80/853 loss=1.8150
  Epoch 4 step 90/853 loss=1.8140
  Epoch 4 step 100/853 loss=1.7961
  Epoch 4 step 110/853 loss=1.7876
  Epoch 4 step 120/853 loss=1.7771
  Epoch 4 step 130/853 loss=1.7827
  Epoch 4 step 140/853 loss=1.7824
  Epoch 4 step 150/853 loss=1.7769
  Epoch 4 step 160/853 loss=1.7730
  Epoch 4 step 170/853 loss=1.7679
  Epoch 4 step 180/853 loss=1.7707
  Epoch 4 step 190/853 loss=1.7722
  Epoch 4 step 200/853 loss=1.7712
  Epoch 4 step 210/853 loss=1.7684
  Epoch 4 step 220/853 loss=1.7656
  Epoch 4 step 230/853 loss=1.7684
  Epoch 4 step 240/853 loss=1.7658
  Epoch 4 step 250/853 loss=1.7683
  Epoch 4 step 260/853 loss=1.7629
  Epoch 4 step 270/853 loss=1.7611
  E

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint to dialogue-augmented-t5/checkpoint-epoch-4
  Epoch 5 step 10/853 loss=1.6716
  Epoch 5 step 20/853 loss=1.6564
  Epoch 5 step 30/853 loss=1.6421
  Epoch 5 step 40/853 loss=1.6495
  Epoch 5 step 50/853 loss=1.6364
  Epoch 5 step 60/853 loss=1.6286
  Epoch 5 step 70/853 loss=1.6370
  Epoch 5 step 80/853 loss=1.6392
  Epoch 5 step 90/853 loss=1.6426
  Epoch 5 step 100/853 loss=1.6359
  Epoch 5 step 110/853 loss=1.6306
  Epoch 5 step 120/853 loss=1.6324
  Epoch 5 step 130/853 loss=1.6317
  Epoch 5 step 140/853 loss=1.6289
  Epoch 5 step 150/853 loss=1.6267
  Epoch 5 step 160/853 loss=1.6324
  Epoch 5 step 170/853 loss=1.6285
  Epoch 5 step 180/853 loss=1.6275
  Epoch 5 step 190/853 loss=1.6224
  Epoch 5 step 200/853 loss=1.6214
  Epoch 5 step 210/853 loss=1.6185
  Epoch 5 step 220/853 loss=1.6136
  Epoch 5 step 230/853 loss=1.6127
  Epoch 5 step 240/853 loss=1.6106
  Epoch 5 step 250/853 loss=1.6107
  Epoch 5 step 260/853 loss=1.6143
  Epoch 5 step 270/853 loss=1.6130
  E

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint to dialogue-augmented-t5/checkpoint-epoch-5

Training complete.
Best val loss: 3.5641
Best model saved to dialogue-augmented-t5/best


In [15]:
runtime.unassign()